In [ ]:
import pandas as pd, hashlib, json
from pathlib import Path

root = Path('.')
df = pd.read_csv(root / 'events_raw.csv')
print('Loaded rows:', len(df))
df.head(3)

## Simple schema validation
We verify required columns exist and values are within simple ranges. In production, you'd use more robust validators (e.g., `jsonschema`).

In [ ]:
required_cols = [
  'timestamp','user_id','role','resource_type','action','bytes_transferred',
  'hour_of_day','geo_region','ip_reputation_score','session_length_sec','anomaly_type'
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

assert df['hour_of_day'].between(0,23).all(), 'hour_of_day out of range'
assert df['ip_reputation_score'].between(0,1).all(), 'ip_reputation_score out of range'
assert (df['bytes_transferred']>=0).all(), 'negative bytes'
print('Basic schema checks passed')

## Build and save a simple manifest
We capture row counts, category frequencies for a few fields, basic numeric summaries, and a SHA256 hash of the CSV.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f: h.update(f.read())
    return h.hexdigest()

manifest = {
  'rows': int(len(df)),
  'freq_role': df['role'].value_counts(normalize=True).to_dict(),
  'freq_resource_type': df['resource_type'].value_counts(normalize=True).to_dict(),
  'numeric_summary': df[['bytes_transferred','ip_reputation_score','session_length_sec']].describe().to_dict(),
  'file': 'events_raw.csv',
  'sha256': sha256(root/'events_raw.csv')
}
with open(root/'manifest_raw.json','w') as f: json.dump(manifest,f,indent=2)
print('Wrote manifest_raw.json')
manifest['sha256'][:16] + '…'